# Partie II — Réseaux convolutifs (CNN) et vision par ordinateur
### Classification d'images — *Fashion-MNIST*

**Projet de fin de module — Deep Learning — EMSI Casablanca, 2025–2026**

---

Notebook **autonome** : import des données, configuration, théorie, calculs
dimensionnels, implémentations « maison », modèle CNN, expériences comparatives
et analyse y figurent. **Aucune valeur n'est codée en dur** : tout passe par la
cellule `Config`.

**Plan**
1. Pourquoi un MLP est mal adapté aux images ; idées fondatrices des CNN
2. Calculs manuels : corrélation croisée, taille de sortie en convolution et après pooling
3. Implémentations « maison » : corrélation croisée 2D, max-pooling, average-pooling, et comparaison aux couches PyTorch
4. Import et préparation de Fashion-MNIST
5. Modèle CNN inspiré de LeNet
6. Étude expérimentale (padding, stride, pooling, nombre de filtres, conv 1×1)
7. Visualisation et interprétation des cartes de caractéristiques
8. Comparaison MLP vs CNN sur le même dataset
9. Analyse critique et question de synthèse


## 1. Pourquoi un MLP est-il mal adapté aux images ?

Une image $28\times28$ aplatie donne un vecteur de 784 entrées. Une première
couche dense de 256 neurones coûte déjà $784\times256 \approx 200\,000$
paramètres, et ce **sans exploiter la structure spatiale**. Trois problèmes :

1. **Explosion paramétrique** — chaque pixel est connecté à chaque neurone.
2. **Perte de la localité** — l'aplatissement détruit le voisinage 2D : deux
   pixels adjacents deviennent deux coordonnées quelconques.
3. **Absence d'invariance** — un objet décalé de quelques pixels active des
   entrées totalement différentes ; le MLP doit tout réapprendre.

### Idées fondatrices des CNN
- **Localité** : un neurone ne regarde qu'une petite fenêtre (champ réceptif).
- **Partage des poids** : le **même** noyau glisse sur toute l'image → invariance
  par translation et chute drastique du nombre de paramètres.
- **Hiérarchie des représentations** : les premières couches détectent des
  bords/textures, les suivantes composent des motifs de plus en plus abstraits.

La **corrélation croisée 2D** est l'opération centrale : un noyau $K$ glisse sur
l'entrée $X$ et produit, en chaque position, une somme pondérée locale.


## 2. Calculs manuels (dimensions)

**Taille de sortie d'une convolution / corrélation croisée.** Pour une entrée
de taille $H\times W$, un noyau $k_h\times k_w$, un *padding* $p$ et un *stride*
$s$ :
$$ H_{out} = \left\lfloor \frac{H + 2p - k_h}{s} \right\rfloor + 1, \qquad
   W_{out} = \left\lfloor \frac{W + 2p - k_w}{s} \right\rfloor + 1. $$

**Exemples (à vérifier numériquement plus bas).**
- $X$ de $28\times28$, $K$ de $5\times5$, $p=0$, $s=1$ → $H_{out}=\lfloor(28-5)/1\rfloor+1 = 24$.
- $X$ de $28\times28$, $K$ de $5\times5$, $p=2$, $s=1$ → $28$ (convolution « same »).
- $X$ de $28\times28$, $K$ de $3\times3$, $p=1$, $s=2$ → $\lfloor(28+2-3)/2\rfloor+1 = 14$.

**Pooling.** Même formule avec la taille de fenêtre comme noyau. Un max-pool
$2\times2$, $s=2$, $p=0$ sur $24\times24$ donne $\lfloor(24-2)/2\rfloor+1 = 12$.

**Corrélation croisée — exemple à la main.** Avec
$X=\begin{bmatrix}0&1&2\\3&4&5\\6&7&8\end{bmatrix}$ et
$K=\begin{bmatrix}0&1\\2&3\end{bmatrix}$, la sortie $2\times2$ vaut
$\begin{bmatrix}19&25\\37&43\end{bmatrix}$, par exemple
$(0\cdot0+1\cdot1+2\cdot3+3\cdot4)=19$.


In [ ]:
import math

def conv_out_size(h, k, p=0, s=1):
    return (h + 2 * p - k) // s + 1

print("28,k5,p0,s1 ->", conv_out_size(28, 5, 0, 1))   # 24
print("28,k5,p2,s1 ->", conv_out_size(28, 5, 2, 1))   # 28
print("28,k3,p1,s2 ->", conv_out_size(28, 3, 1, 2))   # 14
print("pool 24,k2,s2 ->", conv_out_size(24, 2, 0, 2)) # 12

## 3. Implémentations « maison » et comparaison à PyTorch

On programme la corrélation croisée 2D et le pooling avec des boucles
explicites, puis on vérifie qu'elles coïncident avec `F.conv2d` /
`F.max_pool2d` / `F.avg_pool2d`.

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

def corr2d(X, K):
    """Corrélation croisée 2D (entrée et noyau 2D, stride=1, sans padding)."""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

X = torch.arange(9.).reshape(3, 3)
K = torch.tensor([[0., 1.], [2., 3.]])
print("corr2d maison :\n", corr2d(X, K))

# Vérification via F.conv2d (conv2d effectue une corrélation croisée en deep learning)
ref = F.conv2d(X[None, None], K[None, None])  # ajoute dims (batch, canal)
print("F.conv2d       :\n", ref[0, 0])
print("Identiques :", torch.allclose(corr2d(X, K), ref[0, 0]))

In [ ]:
def pool2d(X, size, mode="max"):
    """Pooling 2D maison (stride = taille de fenêtre, sans chevauchement)."""
    ph, pw = size
    H, W = X.shape[0] // ph, X.shape[1] // pw
    Y = torch.zeros((H, W))
    for i in range(H):
        for j in range(W):
            window = X[i * ph:(i + 1) * ph, j * pw:(j + 1) * pw]
            Y[i, j] = window.max() if mode == "max" else window.mean()
    return Y

Xp = torch.arange(16.).reshape(4, 4)
print("max-pool maison :\n", pool2d(Xp, (2, 2), "max"))
print("max-pool PyTorch:\n", F.max_pool2d(Xp[None, None], 2)[0, 0])
print("avg-pool maison :\n", pool2d(Xp, (2, 2), "avg"))
print("avg-pool PyTorch:\n", F.avg_pool2d(Xp[None, None], 2)[0, 0])

## 4. Configuration et import de Fashion-MNIST

Fashion-MNIST : 70 000 images $28\times28$ en niveaux de gris, 10 classes de
vêtements. Chargé via `kagglehub` (CSV `fashion-mnist_train.csv` /
`fashion-mnist_test.csv`), avec repli sur fichiers locaux. Pour rester
exécutable sur CPU, des sous-échantillons (réglables) sont prélevés.

In [ ]:
import os, sys, subprocess, random, json, glob
from dataclasses import dataclass, field, asdict
from typing import List

def _ensure(pkg, imp=None):
    import importlib
    try: importlib.import_module(imp or pkg)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
for _p, _i in [("numpy","numpy"),("pandas","pandas"),("matplotlib","matplotlib"),
               ("seaborn","seaborn"),("scikit-learn","sklearn"),("torch","torch"),("kagglehub","kagglehub")]:
    _ensure(_p, _i)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")

@dataclass
class Config:
    seed: int = 42
    n_train: int = 12000        # sous-échantillon d'entraînement (CPU-friendly)
    n_test: int = 2000          # sous-échantillon de test
    val_frac: float = 0.15
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 0.0
    epochs: int = 6
    exp_epochs: int = 4         # budget réduit pour les études comparatives
    kaggle_id: str = "zalando-research/fashionmnist"
    data_dir: str = os.path.join("..", "data", "fashion_mnist")
    artifacts_dir: str = os.path.join("..", "artifacts", "part2_cnn")

CFG = Config()
os.makedirs(CFG.data_dir, exist_ok=True); os.makedirs(CFG.artifacts_dir, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def try_gpu(i=0):
    return torch.device(f"cuda:{i}") if torch.cuda.device_count() >= i+1 else torch.device("cpu")
set_seed(CFG.seed); DEVICE = try_gpu()
print("Device :", DEVICE); print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))

In [ ]:
CLASSES = ["T-shirt/top","Trouser","Pullover","Dress","Coat",
           "Sandal","Shirt","Sneaker","Bag","Ankle boot"]

def find_file(root, patterns):
    for pat in patterns:
        hits = glob.glob(os.path.join(root, "**", pat), recursive=True)
        if hits: return hits[0]
    return None

def load_fmnist_csv(cfg):
    root = cfg.data_dir
    train_csv = find_file(root, ["fashion-mnist_train.csv"])
    test_csv  = find_file(root, ["fashion-mnist_test.csv"])
    if train_csv is None or test_csv is None:
        import kagglehub
        root = kagglehub.dataset_download(cfg.kaggle_id)
        train_csv = find_file(root, ["fashion-mnist_train.csv"])
        test_csv  = find_file(root, ["fashion-mnist_test.csv"])
    print("Train CSV:", train_csv, "\nTest  CSV:", test_csv)
    return pd.read_csv(train_csv), pd.read_csv(test_csv)

df_train, df_test = load_fmnist_csv(CFG)
print("Brut :", df_train.shape, df_test.shape)

In [ ]:
def df_to_arrays(df):
    y = df["label"].values.astype(np.int64)
    X = df.drop(columns=["label"]).values.astype(np.float32) / 255.0   # normalisation [0,1]
    X = X.reshape(-1, 1, 28, 28)                                       # (N, C=1, 28, 28)
    return X, y

Xtr_full, ytr_full = df_to_arrays(df_train)
Xte_full, yte_full = df_to_arrays(df_test)

# Sous-échantillonnage stratifié (reproductible) pour rester exécutable sur CPU
def subsample(X, y, n, seed):
    if n >= len(X): return X, y
    rng = np.random.default_rng(seed)
    idx = []
    per = n // len(np.unique(y))
    for c in np.unique(y):
        ci = np.where(y == c)[0]
        idx.extend(rng.choice(ci, size=min(per, len(ci)), replace=False))
    idx = np.array(idx); rng.shuffle(idx)
    return X[idx], y[idx]

Xtr, ytr = subsample(Xtr_full, ytr_full, CFG.n_train, CFG.seed)
Xte, yte = subsample(Xte_full, yte_full, CFG.n_test, CFG.seed)

# Découpe train/val
n_val = int(len(Xtr) * CFG.val_frac)
Xval, yval = Xtr[:n_val], ytr[:n_val]
Xtr2, ytr2 = Xtr[n_val:], ytr[n_val:]
print(f"Train={Xtr2.shape} Val={Xval.shape} Test={Xte.shape}")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
def loader(X, y, bs, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)
train_loader = loader(Xtr2, ytr2, CFG.batch_size, True)
val_loader   = loader(Xval, yval, CFG.batch_size, False)
test_loader  = loader(Xte,  yte,  CFG.batch_size, False)

# Aperçu de quelques images
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, i in zip(axes.ravel(), range(10)):
    ax.imshow(Xtr2[i, 0], cmap="gray"); ax.set_title(CLASSES[ytr2[i]], fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Modèle CNN inspiré de LeNet

Architecture paramétrable (filtres, pooling, padding, stride, conv 1×1) afin de
servir aussi aux études comparatives de la section 6. La déduction automatique
de la taille aplatie (via une passe à blanc) évite tout calcul codé en dur.

In [ ]:
class LeNetLike(nn.Module):
    def __init__(self, num_classes=10, c1=6, c2=16,
                 padding=2, stride=1, pool="max", use_1x1=False):
        super().__init__()
        Pool = nn.MaxPool2d if pool == "max" else nn.AvgPool2d
        layers = [nn.Conv2d(1, c1, kernel_size=5, padding=padding, stride=stride),
                  nn.ReLU(), Pool(2)]
        if use_1x1:                       # conv 1x1 : mélange de canaux, +non-linéarité
            layers += [nn.Conv2d(c1, c1, kernel_size=1), nn.ReLU()]
        layers += [nn.Conv2d(c1, c2, kernel_size=5, padding=padding, stride=stride),
                   nn.ReLU(), Pool(2)]
        self.features = nn.Sequential(*layers)
        # Déduction dynamique de la dimension aplatie
        with torch.no_grad():
            flat = self.features(torch.zeros(1, 1, 28, 28)).numel()
        self.classifier = nn.Sequential(nn.Flatten(),
                                        nn.Linear(flat, 120), nn.ReLU(),
                                        nn.Linear(120, 84), nn.ReLU(),
                                        nn.Linear(84, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
demo = LeNetLike()
print(demo)
print("Paramètres :", count_params(demo))

In [ ]:
def train_clf(model, tr_loader, va_loader, cfg, device, epochs=None, verbose=False):
    model.to(device); epochs = epochs or cfg.epochs
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "val_acc": []}
    best_acc, best_state = 0.0, None
    for ep in range(epochs):
        model.train(); run = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(model(xb), yb)
            loss.backward(); opt.step(); run += loss.item() * xb.size(0)
        acc = clf_accuracy(model, va_loader, device)
        hist["train_loss"].append(run / len(tr_loader.dataset)); hist["val_acc"].append(acc)
        if acc > best_acc:
            best_acc, best_state = acc, {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose: print(f"  ép {ep+1}/{epochs} | loss {hist['train_loss'][-1]:.4f} | val_acc {acc:.4f}")
    if best_state: model.load_state_dict(best_state)
    hist["best_val_acc"] = best_acc
    return hist

@torch.no_grad()
def clf_accuracy(model, loader, device):
    model.eval(); correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item(); total += yb.size(0)
    return correct / total

@torch.no_grad()
def predict_all(model, loader, device):
    model.eval(); ys, ps = [], []
    for xb, yb in loader:
        ps.append(model(xb.to(device)).argmax(1).cpu()); ys.append(yb)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

In [ ]:
set_seed(CFG.seed)
cnn = LeNetLike()
hist_cnn = train_clf(cnn, train_loader, val_loader, CFG, DEVICE, verbose=True)
torch.save(cnn.state_dict(), os.path.join(CFG.artifacts_dir, "lenet_baseline.pt"))
print("Accuracy test (CNN baseline) :", clf_accuracy(cnn, test_loader, DEVICE))

## 6. Étude expérimentale des choix architecturaux

On fait varier **un facteur à la fois** autour de la configuration de base, à
budget d'époques réduit (`CFG.exp_epochs`), et on compare l'accuracy de
validation. Cette approche « toutes choses égales par ailleurs » isole l'effet
de chaque choix.

In [ ]:
def run_variant(name, **kwargs):
    set_seed(CFG.seed)
    m = LeNetLike(**kwargs)
    h = train_clf(m, train_loader, val_loader, CFG, DEVICE, epochs=CFG.exp_epochs)
    return {"variante": name, "val_acc": round(h["best_val_acc"], 4),
            "params": count_params(m)}

experiments = [
    run_variant("base (pad=2,s=1,max,c=6/16)"),
    run_variant("padding=0",        padding=0),
    run_variant("stride=2",         stride=2),
    run_variant("avg-pooling",      pool="avg"),
    run_variant("plus de filtres",  c1=16, c2=32),
    run_variant("conv 1x1",         use_1x1=True),
]
df_exp = pd.DataFrame(experiments).set_index("variante")
df_exp

In [ ]:
ax = df_exp["val_acc"].plot(kind="barh", figsize=(8, 4), color="steelblue")
ax.set_xlabel("accuracy de validation"); ax.set_title("Effet des choix architecturaux")
ax.set_xlim(df_exp["val_acc"].min() - 0.03, df_exp["val_acc"].max() + 0.01)
for i, v in enumerate(df_exp["val_acc"]): ax.text(v + 0.001, i, f"{v:.3f}", va="center")
plt.tight_layout(); plt.show()

**Interprétation.**
- **padding=0** réduit la carte spatiale et rogne les bords : souvent une légère
  perte d'information utile aux frontières.
- **stride=2** sous-échantillonne fortement dès la convolution : moins de calcul,
  mais résolution spatiale dégradée → accuracy généralement en baisse.
- **avg- vs max-pooling** : le max conserve les activations les plus saillantes
  (bords nets), l'average lisse — sur Fashion-MNIST le max est souvent
  légèrement supérieur.
- **plus de filtres** augmente la capacité (et le coût) : gain si le sous-
  échantillon ne provoque pas de sur-apprentissage.
- **conv 1×1** recombine les canaux à moindre coût et ajoute une non-linéarité ;
  son effet est modéré sur une si petite architecture mais illustre le mécanisme
  des « réseaux dans le réseau ».

## 7. Visualisation des cartes de caractéristiques

On récupère, via un *forward hook*, les activations de la première couche de
convolution pour une image, afin d'observer les motifs (bords, contours)
détectés.

In [ ]:
activations = {}
def hook(module, inp, out): activations["conv1"] = out.detach().cpu()

first_conv = cnn.features[0]
h = first_conv.register_forward_hook(hook)
sample = torch.from_numpy(Xte[0:1]).to(DEVICE)
_ = cnn(sample)
h.remove()

fmap = activations["conv1"][0]      # (C, H, W)
n = fmap.shape[0]
fig, axes = plt.subplots(1, n + 1, figsize=(2 * (n + 1), 2.2))
axes[0].imshow(Xte[0, 0], cmap="gray"); axes[0].set_title(f"entrée\n{CLASSES[yte[0]]}", fontsize=8)
axes[0].axis("off")
for k in range(n):
    axes[k + 1].imshow(fmap[k], cmap="viridis"); axes[k + 1].set_title(f"filtre {k}", fontsize=8)
    axes[k + 1].axis("off")
plt.suptitle("Cartes de caractéristiques — 1ère convolution"); plt.tight_layout(); plt.show()

On observe que **différents filtres réagissent à différentes orientations de
bords** : certains soulignent les contours verticaux, d'autres les horizontaux
ou les zones de transition. C'est la matérialisation de la **hiérarchie des
représentations** : ces détecteurs bas niveau seront recombinés par les couches
suivantes en motifs plus complexes.

## 8. Comparaison MLP vs CNN sur le même dataset

À budget d'entraînement comparable, on oppose un MLP dense (qui ignore la
structure 2D) au CNN.

In [ ]:
class MLPImage(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(),
                                 nn.Linear(28 * 28, 256), nn.ReLU(), nn.Dropout(0.2),
                                 nn.Linear(256, 128), nn.ReLU(),
                                 nn.Linear(128, num_classes))
    def forward(self, x): return self.net(x)

set_seed(CFG.seed)
mlp_img = MLPImage()
hist_mlp = train_clf(mlp_img, train_loader, val_loader, CFG, DEVICE, verbose=False)

comp = pd.DataFrame([
    {"modèle": "MLP dense", "acc_test": clf_accuracy(mlp_img, test_loader, DEVICE),
     "params": count_params(mlp_img)},
    {"modèle": "CNN (LeNet)", "acc_test": clf_accuracy(cnn, test_loader, DEVICE),
     "params": count_params(cnn)},
]).set_index("modèle")
comp.round(4)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
yt, yp = predict_all(cnn, test_loader, DEVICE)
print(classification_report(yt, yp, target_names=CLASSES))
cm = confusion_matrix(yt, yp)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
plt.xlabel("Prédit"); plt.ylabel("Réel"); plt.title("Matrice de confusion — CNN")
plt.tight_layout(); plt.show()

## 9. Analyse critique

- **CNN > MLP.** À nombre de paramètres souvent **inférieur**, le CNN dépasse le
  MLP en accuracy : le partage de poids et la localité encodent un *a priori*
  adapté aux images, là où le MLP doit tout réapprendre depuis des pixels
  aplatis.
- **Confusions résiduelles.** Les erreurs se concentrent entre classes
  visuellement proches (*Shirt* vs *T-shirt/top* vs *Pullover* vs *Coat*) :
  leurs silhouettes en niveaux de gris se ressemblent, ce qu'aucune
  architecture ne lève totalement à cette résolution.
- **Dimensionnel.** Les choix de padding/stride/pooling ont un effet **mesurable
  et prévisible** par les formules de la section 2 : ils gouvernent la taille des
  cartes, donc le champ réceptif et le coût.
- **Limites.** Budget volontairement réduit (sous-échantillon, peu d'époques)
  pour la portabilité CPU ; augmenter `n_train`/`epochs` dans `Config`
  rapproche des ~92 % typiques d'un LeNet sur Fashion-MNIST complet. Pas
  d'augmentation de données ni de *batch-norm* ici, qui amélioreraient encore.

### Question de synthèse — Partie II
> *Pourquoi un CNN est-il plus pertinent qu'un MLP pour la classification
> d'images, et comment padding, stride, pooling et profondeur influencent-ils
> les performances ?*

Un CNN est plus pertinent parce qu'il **encode la structure spatiale** que le
MLP détruit : la **localité** (champs réceptifs restreints) et le **partage des
poids** (un noyau glissant) procurent une invariance par translation et
réduisent massivement le nombre de paramètres, tout en empilant une
**hiérarchie** de détecteurs (bords → motifs → objets). Nos expériences le
confirment : à paramètres comparables ou inférieurs, le CNN surpasse le MLP
dense. Les **choix dimensionnels** agissent de façon quantifiable via
$H_{out}=\lfloor(H+2p-k)/s\rfloor+1$ : le *padding* préserve l'information de
bord et la taille spatiale ; le *stride* sous-échantillonne (gain de calcul mais
perte de résolution) ; le *pooling* introduit une invariance locale (le max
retient les activations saillantes, l'average lisse) ; la *profondeur* et le
**nombre de filtres** augmentent le champ réceptif effectif et la capacité
représentationnelle, au prix d'un coût accru et d'un risque de sur-apprentissage.
La convolution **1×1** illustre enfin la recombinaison de canaux à faible coût.
Les performances résultent donc d'un compromis réglé par ces hyperparamètres
entre résolution spatiale, capacité et coût.
